# Qwen LoRA SFT / Data Pipeline / DPO Learning Notebook

这个 notebook 是本项目的“可操作学习地图”。它把命令行脚本串成可以逐格运行的流程，帮助理解：

- Qwen base 推理是什么样子。
- LoRA SFT 怎样准备数据、训练 adapter、加载 adapter。
- 为什么公开通用数据集能跑通工程闭环，但不一定修正 LoRA/SFT/DPO 技术概念误解。
- 后续 Stage 2B 如何做自采集数据、清洗、转换。
- DPO 为什么更吃显存，以及 8GB 显存下如何保守测试。

当前 notebook 先覆盖到 Stage 4A。Stage 2B、Stage 3B、DPO 部分先写学习骨架，后续边做边补。

## 0. 使用方式

建议用 `qwen-lora-local` 环境打开这个 notebook。大多数 cell 是轻量检查；训练 cell 默认不会执行，需要手动把开关改成 `True`。

如果你只是复盘项目，按顺序运行环境检查、推理、报告读取即可。如果你要重新训练，再打开训练开关。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

os.environ["HF_HOME"] = str((PROJECT_ROOT / ".hf_cache").resolve())
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("HF_HOME:", os.environ["HF_HOME"])

def run_cmd(args, timeout=None):
    print("$", " ".join(str(x) for x in args))
    result = subprocess.run(
        [str(x) for x in args],
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        timeout=timeout,
    )
    print(result.stdout)
    if result.stderr:
        print("[stderr]")
        print(result.stderr)
    result.check_returncode()
    return result

## 1. 环境检查

先确认 Python 包版本和 CUDA。这个项目之前在 Windows 原生环境里遇到过高版本 Hugging Face 栈导致 `python.exe` 进程级崩溃的问题，所以环境版本是项目经验的一部分。

In [ ]:
run_cmd([sys.executable, "scripts/check_env.py"], timeout=60)
run_cmd([
    sys.executable,
    "-c",
    "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')",
], timeout=60)

## 2. 先讲清楚术语

- SFT 是 supervised fine-tuning，指训练目标和数据形式。
- LoRA 是 parameter-efficient fine-tuning 方法，指只训练 adapter 小矩阵。
- LoRA SFT 的意思是：用 LoRA adapter 来做 SFT。
- DPO 是 direct preference optimization，使用 chosen/rejected 偏好对进一步调模型行为。

Stage 4A 的重要发现是：公开通用数据集虽然跑通了 LoRA SFT，但没有修正这些技术概念误解。

## 3. Base 模型推理

先看没有 adapter 的 Qwen 怎么回答。这是后续对比的 baseline。

In [ ]:
run_cmd([
    sys.executable,
    "scripts/infer.py",
    "--prompt", "请用三点解释什么是LoRA微调。",
    "--max_new_tokens", "96",
    "--local_files_only",
], timeout=180)

## 4. Stage 2A：公开数据集准备

当前公开数据集已经准备好：

- raw: `data/raw/alpaca_gpt4_data_zh_1200.jsonl`
- train: `data/processed/sft_train.jsonl`
- eval: `data/processed/sft_eval.jsonl`

如果以后要重跑，把下面的 `RUN_PUBLIC_DATA_PREP` 改成 `True`。

In [ ]:
RUN_PUBLIC_DATA_PREP = False

if RUN_PUBLIC_DATA_PREP:
    run_cmd([
        sys.executable,
        "scripts/download_hf_sft_data.py",
        "--dataset_name", "llm-wizard/alpaca-gpt4-data-zh",
        "--split", "train",
        "--output_file", "data/raw/alpaca_gpt4_data_zh_1200.jsonl",
        "--max_samples", "1200",
        "--seed", "42",
    ], timeout=600)
    run_cmd([
        sys.executable,
        "scripts/prepare_data.py",
        "--input_file", "data/raw/alpaca_gpt4_data_zh_1200.jsonl",
        "--train_file", "data/processed/sft_train.jsonl",
        "--eval_file", "data/processed/sft_eval.jsonl",
        "--eval_ratio", "0.1",
        "--max_samples", "1200",
        "--min_answer_chars", "20",
        "--seed", "42",
    ], timeout=120)
else:
    for path in ["data/processed/sft_train.jsonl", "data/processed/sft_eval.jsonl"]:
        rows = [line for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
        print(path, "rows =", len(rows))

## 5. Stage 3A：公开数据 LoRA SFT

这一步训练耗时约 24 分钟，训练时显存观察约 `5.5GB / 8GB`。默认不执行。

已经完成的 adapter：`outputs/sft_lora_qwen05b_public`。

In [ ]:
RUN_PUBLIC_SFT_TRAINING = False

if RUN_PUBLIC_SFT_TRAINING:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--train_file", "data/processed/sft_train.jsonl",
        "--eval_file", "data/processed/sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_public",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "8",
        "--epochs", "1",
        "--logging_steps", "10",
        "--eval_steps", "100",
        "--save_steps", "100",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)
else:
    adapter_dir = Path("outputs/sft_lora_qwen05b_public")
    print("Adapter exists:", adapter_dir.exists())
    print("Adapter files:", [p.name for p in adapter_dir.glob("adapter_*")])

## 6. Stage 4A：Base vs Public-SFT 对比

这一步是现在最重要的结论来源：公开数据 SFT 能训练成功，但没有修正 LoRA/SFT/DPO 的概念误解。

In [ ]:
RUN_STAGE4A_COMPARE = False

if RUN_STAGE4A_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_outputs.py",
        "--prompt_file", "data/samples/smoke_prompts.jsonl",
        "--adapter_path", "outputs/sft_lora_qwen05b_public",
        "--output_file", "reports/compare_outputs_public_sft.jsonl",
        "--max_new_tokens", "96",
        "--local_files_only",
    ], timeout=900)

rows = [json.loads(line) for line in Path("reports/compare_outputs_public_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
for row in rows:
    print("=" * 80)
    print("Prompt:", row["prompt"])
    print("\nBase:\n", row["base_answer"][:500])
    print("\nPublic-SFT:\n", row["sft_answer"][:500])

## 7. Stage 4A 结论

公开通用数据集的价值：

- 证明了数据转换、tokenizer、LoRA 训练、adapter 保存/加载都能工作。
- 得到了一个可比较的 public-SFT baseline。

公开通用数据集的不足：

- 没有修正 LoRA/SFT/DPO 的目标技术概念。
- LoRA 仍可能被解释成通信 LoRa。
- SFT/DPO 仍可能被解释成无关缩写。

所以 Stage 2B 不是锦上添花，而是必要步骤：我们需要自采集技术数据来教模型目标概念和项目叙述方式。

## 8. Stage 2B：自采集技术数据计划

第一版不要贪多，目标是 100-300 条高质量样本。

优先主题：

- LoRA、SFT、DPO 的概念解释。
- Hugging Face / PEFT / Transformers 使用经验。
- Windows + CUDA 调试经验。
- 训练日志、loss、显存、batch size、gradient accumulation 的解释。
- 面试中如何讲清楚一个本地大模型微调项目。

注意：爬虫要合法、克制。优先使用公开文档、自己的笔记、短摘录、摘要，不要复制整篇受版权保护文章。

In [ ]:
# Stage 2B skeleton: 后续会把这里扩展成真正的爬取脚本或数据收集脚本。
# 当前先定义目标 schema，方便理解每个阶段会产出什么。

custom_source_schema = {
    "source_id": "example-001",
    "url": "https://example.com/article",
    "title": "示例标题",
    "raw_text": "网页或笔记的原始文本",
    "collected_at": "2026-05-14",
}

cleaned_chunk_schema = {
    "source_id": "example-001",
    "chunk_id": "example-001-0001",
    "cleaned_text": "去掉导航、广告、重复空白后的文本块",
    "quality_note": "accepted / rejected reason",
}

instruction_seed_schema = {
    "instruction": "用三点解释 LoRA 为什么适合个人 GPU 学习。",
    "input": "",
    "output": "第一，LoRA 只训练少量 adapter 参数...",
    "source_id": "example-001",
}

print(json.dumps(custom_source_schema, ensure_ascii=False, indent=2))
print(json.dumps(cleaned_chunk_schema, ensure_ascii=False, indent=2))
print(json.dumps(instruction_seed_schema, ensure_ascii=False, indent=2))

## 9. 清洗规则草案

后续真正做 Stage 2B 时，清洗至少包含：

- 去导航栏、页脚、广告、cookie banner。
- 合并多余空白。
- 删除过短、过长、低信息密度文本。
- 精确去重和近似去重。
- 保留来源信息，方便解释数据从哪里来。
- 人工抽查 accepted/rejected 样本。

In [ ]:
import re

def simple_clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = text.replace("Cookie", "")
    return text.strip()

sample_dirty_text = """
首页  导航  Cookie
LoRA 是一种参数高效微调方法。它冻结大部分基础模型参数，只训练少量 adapter 参数。
页脚  联系我们
"""

print(simple_clean_text(sample_dirty_text))

## 10. DPO 和显存预判

当前观察：

- LoRA SFT 训练约 `5.5GB / 8GB`。
- Adapter 推理约 `1.2GB / 8GB`。

DPO 会更吃显存，因为它通常需要 chosen/rejected 两个回答，并且要和 reference policy 对比。8GB 不是完全不能做，但第一版必须非常保守：小模型、LoRA、batch size 1、短序列、少量 pair、少 eval。

In [ ]:
print(Path("reports/vram_and_dpo_plan.md").read_text(encoding="utf-8")[:2000])

## 11. 当前验收清单

- Stage 0/1：环境和 base 推理完成。
- Stage 2A：公开中文 SFT 数据完成。
- Stage 3A：public LoRA SFT 完成。
- Stage 4A：base vs public-SFT 对比完成，结论是没有修正目标技术概念。
- 下一步：Stage 2B 自采集技术数据。

后续每完成一个阶段，就继续更新这个 notebook。